# Imports

In [0]:
print(1)

In [0]:
%pip install catboost lightgbm optuna holidays

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
%matplotlib inline

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, TimeSeriesSplit, cross_val_predict, cross_val_score, cross_validate
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool

import optuna
from typing import Any
import warnings

import holidays

In [0]:
import plotly.io as pio
pio.renderers.default = "vscode"

# Global Vars

In [0]:
TARGET = "num_sold"
N_SPLITS = 5
SEED = 42
eps = 1e-5

def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

non_relevant_columns = ['id', 'source']

# ordinal_maps = {
#     "gender" : {"male":0, "female":1, "other":2},
#     "internet_access" : {"no":0, "yes":1},
#     "sleep_quality" : {"poor":0, "average":1, "good":2},
#     "facility_rating" : {"low":0, "medium":1, "high":2},
#     "exam_difficulty" : {"easy":0, "moderate":1, "hard":2},
#     "course" : {"ba":0, "b.sc":1, "diploma":2, "b.tech":3, "b.com":4, "bca":5, "bba":6},
#     "study_method" : {"self-study":0, "online videos":1, "group study":2, "mixed":3, "coaching":4},
# }

original_categorical_cols = ['country', 'store', 'product']


# Load Data

In [0]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv('/content/drive/MyDrive/Kaggle Practice/Sticker Sales Forecast/train.csv')
    test_df = pd.read_csv('/content/drive/MyDrive/Kaggle Practice/Sticker Sales Forecast/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  train_df['date'] =  pd.to_datetime(train_df['date'])
  test_df['date'] =  pd.to_datetime(test_df['date'])
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

# Data Splitter

In [0]:
def data_spliter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

# EDA

In [0]:
df_train, df_test, df = load_data(option=ENVIRONMENT)

## Info / describe / etc.

In [0]:
df.head()

In [0]:
df.info()

In [0]:
df.describe()

In [0]:
df.isna().sum()

## Test vs Train date

In [0]:
sns.histplot(df, x='date', hue='source')

## Num Sold overtime

In [0]:
df['date'].dt.year

In [0]:
plt.figure(figsize=(10,6))
hist_data = df_train.copy()
hist_data['year'] = hist_data['date'].dt.year
sns.histplot(
    data=hist_data,
    x='num_sold',
    hue='year',
    bins=30,
    log_scale=True,
    palette='viridis',
    multiple='fill',
)

In [0]:
df_test

In [0]:
sns.lineplot(data=df, x='date', y='num_sold')

In [0]:
plt.figure(figsize=(10,6))
grouped = df_train.groupby([pd.Grouper(key='date', freq='D'), 'product'])['num_sold'].sum().reset_index()
sns.lineplot(data=grouped, x='date', y='num_sold', hue='product')

In [0]:
grouped = df_train.groupby([pd.Grouper(key='date', freq='MS'), 'country'])['num_sold'].sum().reset_index()
sns.lineplot(data=grouped, x='date', y='num_sold', hue='country')

In [0]:
df['product'].unique()

## Country

In [0]:
df['country'].unique()

In [0]:
holidays.KE( years=range(2020, 2027))